# Evaluation summary

One-stop summary of the three evaluation notebooks in this folder:

1. `global_priors_evaluation.ipynb` — does $B$ beat heuristic and marginal baselines on held-out actions?
2. `range_evaluation.ipynb` — does the EM E-step posterior $q(h)$ identify the target's actual hand?
3. `player_thetas_evaluation.ipynb` — does adding $\hat\theta$ on top of $\theta=0$ improve held-out action prediction, and does it generalise off the EM split?

This notebook re-executes the headline numbers from each so they stay in sync with whatever is currently in `artifacts/`. It does not re-derive the metrics — see the source notebooks for the full machinery and per-row diagnostics.

**Splits used (seed=42, fractions 0.5 / 0.3 / 0.2 over `sessions.txt`):**
* train = sessions [33, 31, 32]   → fits $B$ → `global_priors.json` (256 hands)
* EM    = session  [34]           → fits per-player $\hat\theta$ → `player_thetas.json`
* online = sessions [30, 35]      → never seen by $B$ or $\hat\theta$; the only honest test set for both

## Re-execute the headline numbers

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pipeline_common import (
    flatten_hands, read_session_names_file, split_session_names,
    collect_preflop_supervised_rows, collect_postflop_supervised_rows,
    hole_cards_to_hand_class, preflop_decisions_for_hand,
)
from utils.action.preflop import (
    PreflopActionModel, PreflopPrior, HEURISTIC_BETA_PREFLOP, canonical_preflop_action,
    FOLD as PRE_FOLD, CHECK_CALL as PRE_CALL, RAISE as PRE_RAISE,
)
from utils.action.postflop import (
    PostflopActionModel, PostflopPrior, HEURISTIC_BETA_FACING, HEURISTIC_BETA_NO_BET,
    FOLD as POST_FOLD, CALL as POST_CALL, RAISE as POST_RAISE,
)
from utils.em.preflop import PreflopEMDecision, PreflopEMHandBundle, e_step_hand_class_posterior
from utils.postflop_runner_bridge import collect_postflop_observations_known_hole_cards
from utils.filter.common import initial_class_prior, normalize
from utils.strength.preflop import all_169_classes
from utils.eval.brier import multiclass_brier

priors = json.loads((REPO_ROOT / "artifacts" / "global_priors.json").read_text())
BETA_PRE    = np.asarray(priors["preflop"]["beta_preflop"], dtype=float)
BETA_FACING = np.asarray(priors["postflop"]["beta_facing"], dtype=float)
BETA_NO_BET = np.asarray(priors["postflop"]["beta_no_bet"], dtype=float)

thetas_payload = json.loads((REPO_ROOT / "artifacts" / "player_thetas.json").read_text())
PLAYERS    = list(thetas_payload["players"].keys())
THETA_PRE  = {p: tuple(thetas_payload["players"][p]["theta_pre"])  for p in PLAYERS}
THETA_POST = {p: tuple(thetas_payload["players"][p]["theta_post"]) for p in PLAYERS}
OBSERVER_OF = {"Gogo": "Pluribus", "Pluribus": "Gogo"}

session_names = read_session_names_file(REPO_ROOT / "sessions.txt")
train_s, em_s, online_s = split_session_names(
    session_names, train_frac=0.5, em_frac=0.3, online_frac=0.2, seed=42)
train_refs   = flatten_hands([REPO_ROOT / "pluribus" / s for s in train_s])
em_refs      = flatten_hands([REPO_ROOT / "pluribus" / s for s in em_s])
online_refs  = flatten_hands([REPO_ROOT / "pluribus" / s for s in online_s])
heldout_refs = em_refs + online_refs
print(f"train hands: {len(train_refs)}   em: {len(em_refs)}   online: {len(online_refs)}")
print(f"global-prior held-out (em+online): {len(heldout_refs)} hands")

EPS = 1e-12
def softmax_rows(z):
    m = z.max(axis=1, keepdims=True); e = np.exp(z-m); return e/e.sum(axis=1, keepdims=True)
def predict(beta, X): return softmax_rows(X @ beta.T)
def nll(P, y):
    if P.shape[0] == 0: return float("nan")
    return float(-np.log(np.clip(P[np.arange(P.shape[0]), y.astype(int)], EPS, 1)).mean())
def mean_brier(P, y):
    if P.shape[0] == 0: return float("nan")
    return float(np.mean([multiclass_brier(P[i], int(y[i])) for i in range(P.shape[0])]))
def top1(P, y):
    if P.shape[0] == 0: return float("nan")
    return float((P.argmax(axis=1) == y.astype(int)).mean())

### 1) Global priors — held-out (EM + online combined)

In [ ]:
X_pre_train, y_pre_train = collect_preflop_supervised_rows(train_refs)
Xf_train,  yf_train,  Xn_train,  yn_train  = collect_postflop_supervised_rows(train_refs)
X_pre_test,  y_pre_test  = collect_preflop_supervised_rows(heldout_refs)
Xf_test,   yf_test,   Xn_test,   yn_test   = collect_postflop_supervised_rows(heldout_refs)

def remap_no_bet(y):
    if y.size == 0: return y.astype(int)
    return np.where(y == POST_CALL, 0, 1).astype(int)

y_n_train_local = remap_no_bet(yn_train)
y_n_test_local  = remap_no_bet(yn_test)

def marginal(y, k):
    c = np.bincount(y.astype(int), minlength=k).astype(float)
    return np.maximum(c, 1e-12) / np.maximum(c, 1e-12).sum()

marg = {
    "preflop": marginal(y_pre_train, 3),
    "facing":  marginal(yf_train, 3),
    "no_bet":  marginal(y_n_train_local, 2),
}

def constant_probs(p, n): return np.broadcast_to(p, (n, p.size)).copy()

GP_BLOCKS = [
    ("preflop", BETA_PRE,    HEURISTIC_BETA_PREFLOP, X_pre_test,  y_pre_test),
    ("facing",  BETA_FACING, HEURISTIC_BETA_FACING,  Xf_test,     yf_test),
    ("no_bet",  BETA_NO_BET, HEURISTIC_BETA_NO_BET,  Xn_test,     y_n_test_local),
]

print(f"{'head':<8} {'model':<10} {'N':>4} {'NLL':>7} {'Brier':>7} {'top-1':>7}")
for head, beta_t, beta_h, X, y in GP_BLOCKS:
    P_t = predict(beta_t, X)
    P_h = predict(beta_h, X)
    P_m = constant_probs(marg[head], X.shape[0])
    for name, P in (("trained", P_t), ("heuristic", P_h), ("marginal", P_m)):
        print(f"{head:<8} {name:<10} {y.size:>4} {nll(P, y):>7.4f} {mean_brier(P, y):>7.4f} {top1(P, y):>7.4f}")

**Reading.** Trained $B$ wins on every head and metric vs both heuristic and marginal baselines. The wins are larger in NLL/Brier than in top-1 — confusion matrices in the source notebook reveal why: the model collapses to the majority class in argmax (preflop almost always predicts fold; facing-bet *never* predicts raise; no-bet predicts call too often). The fit is **calibrated-better** rather than **classifier-better**, which is exactly the property the downstream Bayes inversion needs.

**Verdict:** sanity check passes; the prior is structurally fit for purpose. Argmax-flatness in rare classes is a small-data artifact, not a feature-spec problem.

### 2) Range posteriors — preflop, per-player, online split

In [ ]:
def build_bundles(refs, target, observer):
    rows = []
    for ref in refs:
        names = ref.hand.player_names
        if target not in names or observer not in names: continue
        true_class = hole_cards_to_hand_class(ref.hand.hole_cards.get(target, "") or "")
        if true_class is None: continue
        decisions = preflop_decisions_for_hand(ref.hand, target, ref.global_index)
        if not decisions: continue
        dead = ref.hand.hole_cards.get(observer, "") or ""
        initial_range = normalize(initial_class_prior(dead_cards=dead))
        bundle = PreflopEMHandBundle(
            tuple(PreflopEMDecision(d.state_key, d.action_bucket) for d in decisions),
            initial_range)
        rows.append((bundle, true_class))
    return rows

ALL169 = all_169_classes()
ALL169_IX = {h: i for i, h in enumerate(ALL169)}

def hand_nll(results):
    if not results: return float("nan")
    return float(np.mean([-np.log(max(float(q.get(tc, 0.0)), EPS)) for tc, q in results]))

def hand_ranks(results):
    out = []
    for tc, q in results:
        p = np.array([q.get(h, 0.0) for h in ALL169], dtype=float)
        order = np.argsort(-p, kind="stable")
        out.append(int(np.where(order == ALL169_IX[tc])[0][0]) + 1)
    return np.asarray(out, dtype=int)

def topk(r, k): return float((r <= k).mean())

print(f"{'player':<10} {'predictor':<11} {'N':>4} {'NLL':>7} {'mean_rk':>8} {'top-10':>7} {'top-50':>7}")
for player in PLAYERS:
    rows = build_bundles(online_refs, player, OBSERVER_OF[player])
    prior_pop = PreflopActionModel(PreflopPrior(beta_preflop=BETA_PRE), (0.0, 0.0, 0.0))
    prior_pl  = PreflopActionModel(PreflopPrior(beta_preflop=BETA_PRE), tuple(THETA_PRE[player]))
    res = {
        "prior_only": [(tc, dict(b.initial_range)) for b, tc in rows],
        "population": [(tc, e_step_hand_class_posterior(b, prior_pop)) for b, tc in rows],
        "player":     [(tc, e_step_hand_class_posterior(b, prior_pl))  for b, tc in rows],
    }
    for name, r_list in res.items():
        r = hand_ranks(r_list)
        print(f"{player:<10} {name:<11} {len(rows):>4} {hand_nll(r_list):>7.3f} "
              f"{r.mean():>8.2f} {topk(r, 10):>7.3f} {topk(r, 50):>7.3f}")

**Reading.** Adding the action sequence (population vs prior_only) drops mean rank from ~60 to ~50 and shaves ~0.1 nats off the true-hand NLL — the EM E-step machinery is working and contributes information about the latent hand.

Adding the **player tilt** (player vs population) on top of that contributes essentially nothing — the two columns are equal to ~3 decimal places. With the small $\hat\theta$ values on file ($|\theta_k| \le 0.08$ preflop), the per-player layer is invisible at the range level.

**Verdict:** if the goal is range inference, the per-player tilt is dead weight at the current data scale. The action-conditioned population posterior is doing all the lifting.

### 3) Player $\hat\theta$ — held-out action prediction (online split)

In [ ]:
def collect_pre(refs, player):
    rows = []
    for ref in refs:
        if player not in ref.hand.player_names: continue
        hc = hole_cards_to_hand_class(ref.hand.hole_cards.get(player, "") or "")
        if hc is None: continue
        for d in preflop_decisions_for_hand(ref.hand, player, ref.global_index):
            rows.append((hc, d.state_key, canonical_preflop_action(d.action_bucket)))
    return rows

def collect_post(refs, player):
    f, n = [], []
    for ref in refs:
        if player not in ref.hand.player_names: continue
        obs = collect_postflop_observations_known_hole_cards(ref.hand, player, ref.global_index)
        if obs is None: continue
        for feat, action in obs.decisions:
            if feat.facing_bet: f.append((feat, int(action)))
            elif int(action) != POST_FOLD: n.append((feat, int(action)))
    return f, n

def to3(d, a, b, c): return np.array([d[a], d[b], d[c]], dtype=float)

def predict_pre(rows, prior):
    if not rows: return np.zeros((0, 3)), np.zeros((0,), dtype=int)
    P = np.zeros((len(rows), 3)); y = np.zeros(len(rows), dtype=int)
    for i, (hc, sk, a) in enumerate(rows):
        P[i] = to3(prior.action_probs(hc, sk), PRE_FOLD, PRE_CALL, PRE_RAISE); y[i] = int(a)
    return P, y
def predict_post_face(rows, prior):
    if not rows: return np.zeros((0, 3)), np.zeros((0,), dtype=int)
    P = np.zeros((len(rows), 3)); y = np.zeros(len(rows), dtype=int)
    for i, (feat, a) in enumerate(rows):
        P[i] = to3(prior.action_probs(feat), POST_FOLD, POST_CALL, POST_RAISE); y[i] = int(a)
    return P, y
def predict_post_nb(rows, prior):
    if not rows: return np.zeros((0, 2)), np.zeros((0,), dtype=int)
    P = np.zeros((len(rows), 2)); y = np.zeros(len(rows), dtype=int)
    for i, (feat, a) in enumerate(rows):
        pr = prior.action_probs(feat)
        P[i, 0], P[i, 1] = pr[POST_CALL], pr[POST_RAISE]
        y[i] = 0 if int(a) == POST_CALL else 1
    return P, y

L2 = 0.25
def heldout_grad(Pt, y, theta_hat, k):
    if Pt.shape[0] == 0: return np.zeros(k)
    g = np.zeros(k)
    for i in range(Pt.shape[0]):
        e = np.zeros(k); e[int(y[i])] = 1.0
        g += e - Pt[i]
    g /= Pt.shape[0]
    g -= L2 * np.asarray(theta_hat[:k])
    return g

print(f"{'player':<10} {'head':<8} {'N':>4} {'NLL_0':>7} {'NLL_θ':>7} {'ΔNLL':>+7} {'flips':>6} {'||g||':>8}")
for player in PLAYERS:
    pre0  = PreflopActionModel(PreflopPrior(beta_preflop=BETA_PRE), (0.0, 0.0, 0.0))
    preH  = PreflopActionModel(PreflopPrior(beta_preflop=BETA_PRE), tuple(THETA_PRE[player]))
    post0 = PostflopActionModel(PostflopPrior(beta_facing=BETA_FACING, beta_no_bet=BETA_NO_BET), (0.0, 0.0, 0.0))
    postH = PostflopActionModel(PostflopPrior(beta_facing=BETA_FACING, beta_no_bet=BETA_NO_BET), tuple(THETA_POST[player]))
    rows_pre = collect_pre(online_refs, player)
    rows_f, rows_n = collect_post(online_refs, player)
    for head, P_pair, theta_hat, k in [
        ("preflop", (predict_pre(rows_pre, pre0),  predict_pre(rows_pre, preH)),  THETA_PRE[player], 3),
        ("facing",  (predict_post_face(rows_f, post0), predict_post_face(rows_f, postH)), THETA_POST[player], 3),
        ("no_bet",  (predict_post_nb(rows_n, post0),   predict_post_nb(rows_n, postH)),   THETA_POST[player], 2),
    ]:
        (P0, y), (Pt, _) = P_pair
        n0, nt = nll(P0, y), nll(Pt, y)
        flips = float((P0.argmax(axis=1) != Pt.argmax(axis=1)).mean()) if P0.shape[0] else float("nan")
        gn = float(np.linalg.norm(heldout_grad(Pt, y, theta_hat, k)))
        print(f"{player:<10} {head:<8} {y.size:>4} {n0:>7.4f} {nt:>7.4f} {n0-nt:>+7.4f} {flips:>6.3f} {gn:>8.5f}")

**Reading.** Two findings:

1. **$\Delta\mathrm{NLL}$ is negative on most heads.** $\hat\theta$ slightly *hurts* held-out action prediction relative to $\theta=0$. Pluribus's facing-bet and no_bet heads are the worst offenders (–0.06 to –0.09 nats). Gogo's tilts are tiny so the change is tiny. The EM-split (in-sample) version of this table shows small *positive* $\Delta\mathrm{NLL}$, so the gap is the standard generalisation gap.
2. **Held-out gradient norms are 0.04–0.39.** The M-step convergence tolerance is $10^{-5}$. A gradient that's 4–5 orders of magnitude above tolerance on held-out rows means $\hat\theta$ sits at a local optimum on its own training data only — it does not generalise. This is the cleanest single-number diagnostic of overfitting.

**Verdict:** the per-player tilt as currently fit is not trustworthy. Acting on $\hat\theta$ at the per-decision level on new hands would slightly degrade prediction. Consistent with finding (2): if it doesn't help action prediction, it can't help the range posterior either.

## Joint diagnosis

The three notebooks form a single story.

* The **prior $B$** is fine: trained beats heuristic and marginal everywhere on probability metrics. Argmax-flatness is a known consequence of $\sim$1–2k samples per softmax head, not a model-spec problem.
* The **EM E-step** is fine: the action-conditioned population posterior narrows the latent-hand distribution from random (~uniform over 169) to mean rank ~50. The mathematical machinery works.
* The **EM M-step** is the weak link. With 78 preflop bundles and 16–18 postflop bundles per player, $\hat\theta$ overfits the EM split and contributes (a) ~zero to range identification and (b) slightly negative value to held-out action prediction.

## Concrete next steps, ranked

1. **More data, by far the highest-leverage move.** Expand `sessions.txt` so the 30% EM split contains hundreds of bundles per player rather than tens. The architecture is fine; it just needs more rows for both the train and EM splits.
2. **Stronger L2 in the M-step.** Bump `m_l2` from 0.25 to ~1.0 in `run_preflop_em` / `run_postflop_theta_em` so $\hat\theta$ shrinks harder toward zero when bundles are scarce. Cheap to try, will improve held-out NLL at the cost of in-sample fit.
3. **L2 + more epochs in the prior fit.** The $B$ matrices are trained with `epochs=50, l2=0.0`. A short rerun with `--preflop-l2 0.05 --preflop-epochs 200` would address the argmax-flatness on rare classes (facing-bet "never predicts raise") without changing anything downstream.
4. **Sanity-check the held-out gradient direction.** The notebook reports the gradient *norm*; eyeballing the per-component direction shows whether the EM-fit $\theta$ is moving toward/away from the held-out optimum. If the directions disagree across the EM and online splits, that is distribution shift between sessions, not just sample noise.

## What we should *not* do yet

* Build downstream consumers (range-filter dashboards, action prediction in a bot loop) that depend on $\hat\theta$ being meaningful per-player. At this scale they will be indistinguishable from the population baseline at best, and slightly worse at worst.
* Tune the $\phi$ feature set or change the prior parameterisation. The diagnostics show the bottleneck is data quantity in the M-step, not feature engineering.